# When the shelf is empty, sales lie: recovering censored demand in fresh retail

**Dataset:** [FreshRetailNet-50K](https://huggingface.co/datasets/Dingdong-Inc/FreshRetailNet-50K) (Dingdong, 2025, CC BY 4.0):
50,000 store-product series of fresh products, 898 stores, 90 days plus a 7-day evaluation split, with **hourly sales and hourly stock-out flags**.

**The problem.** On a stock-out day, recorded sales are only the demand from the hours the product was on the shelf.
Train a forecaster on raw sales and it learns the shortage: it under-forecasts, the store under-orders, and the shelf runs empty again.

**What this notebook does**
1. Explores **when** shelves run empty.
2. **Recovers latent demand** with two methods: hourly-profile scaling, and a *self-supervised* LightGBM trained by hiding hours on in-stock days using real stock-out patterns.
3. **Validates recovery** with a masking test on held-out series, where the hidden demand is known.
4. Trains the same 7-day forecaster on raw vs recovered demand, and evaluates it from **three views of the truth**, because the obvious evaluation (in-stock days only) is itself biased.
5. Sizes the **demand lost** to stock-outs, by category, store and hour.

*Paper:* [FreshRetailNet-50K, arXiv 2505.16319](https://arxiv.org/abs/2505.16319).

In [ ]:
!pip install -q lightgbm huggingface_hub

## The `frn` package
The same code runs the Databricks version of this project.

In [ ]:
import os
os.makedirs('frn', exist_ok=True)
# FreshRetailNet censored-demand toolkit

In [ ]:
%%writefile frn/data.py
"""Load FreshRetailNet-50K into a compact panel: scalar columns in pandas,
the 24-hour sales / out-of-stock lists as dense numpy matrices (n_rows x 24).

Dataset: Dingdong-Inc/FreshRetailNet-50K (Hugging Face, CC BY 4.0)
  data/train.parquet  ~4.5M rows = 50,000 store-product series x 90 days
  data/eval.parquet   ~350k rows = the next 7 days
"""
from __future__ import annotations

import os
import shutil
from dataclasses import dataclass

import numpy as np
import pandas as pd

REPO_ID = "Dingdong-Inc/FreshRetailNet-50K"
FILES = {"train": "data/train.parquet", "eval": "data/eval.parquet"}
KEY = ["store_id", "product_id"]
SCALARS = ["city_id", "store_id", "management_group_id", "first_category_id", "second_category_id",
           "third_category_id", "product_id", "dt", "sale_amount", "stock_hour6_22_cnt", "discount",
           "holiday_flag", "activity_flag", "precpt", "avg_temperature", "avg_humidity", "avg_wind_level"]
OP_HOURS = slice(6, 22)  # 06:00-21:59 (16 h): verified to match stock_hour6_22_cnt on 100% of rows
N_OP_HOURS = OP_HOURS.stop - OP_HOURS.start


@dataclass
class Panel:
    df: pd.DataFrame          # one row per store-product-day
    sales: np.ndarray         # (n, 24) hourly sales
    oos: np.ndarray           # (n, 24) 1 = out of stock in that hour

    def take(self, idx) -> "Panel":
        return Panel(self.df.iloc[idx].reset_index(drop=True), self.sales[idx], self.oos[idx])


def download(dest_dir: str) -> dict[str, str]:
    """Download the two parquet files from Hugging Face into dest_dir (e.g. a UC Volume)."""
    from huggingface_hub import hf_hub_download
    os.makedirs(dest_dir, exist_ok=True)
    out = {}
    for split, fname in FILES.items():
        target = os.path.join(dest_dir, f"{split}.parquet")
        if not os.path.exists(target):
            src = hf_hub_download(REPO_ID, fname, repo_type="dataset")
            shutil.copy(src, target)
        out[split] = target
    return out


def _list_to_matrix(col, width: int = 24) -> np.ndarray:
    import pyarrow as pa
    import pyarrow.compute as pc
    arr = col.combine_chunks() if hasattr(col, "combine_chunks") else col
    lens = pc.list_value_length(arr).to_numpy(zero_copy_only=False)
    flat = np.asarray(arr.flatten().to_numpy(zero_copy_only=False), dtype=float)
    if (lens == width).all():
        return flat.reshape(-1, width)
    out = np.zeros((len(lens), width))          # pad/truncate irregular rows (defensive)
    pos = 0
    for i, n in enumerate(lens):
        out[i, :min(n, width)] = flat[pos:pos + min(n, width)]
        pos += n
    return out


def sample_series(path: str, n_series: int | None, seed: int = 42) -> pd.DataFrame | None:
    """Pick whole stores at random until ~n_series store-product series are covered.
    Sampling by store (not by series) keeps every store's assortment complete and lets the
    parquet reader filter rows while reading, so the full 4.5M-row file never sits in memory."""
    if not n_series:
        return None
    keys = pd.read_parquet(path, columns=KEY).drop_duplicates()
    per_store = keys.groupby("store_id").size().sample(frac=1, random_state=seed)
    stores = per_store.index[: int((per_store.cumsum() < n_series).sum()) + 1]
    return keys[keys["store_id"].isin(stores)]


def load_panel(path: str, keys: pd.DataFrame | None = None) -> Panel:
    import pyarrow as pa
    import pyarrow.parquet as pq
    filters = [("store_id", "in", sorted(keys["store_id"].unique().tolist()))] if keys is not None else None
    t = pq.read_table(path, filters=filters)
    if keys is not None:
        k = t.select(KEY).to_pandas()
        mask = k.merge(keys.assign(_keep=1), on=KEY, how="left")["_keep"].fillna(0).astype(bool).to_numpy()
        t = t.filter(pa.array(mask))
    df = t.select(SCALARS).to_pandas()
    sales = _list_to_matrix(t.column("hours_sale"))
    oos = _list_to_matrix(t.column("hours_stock_status")).astype(np.int8)
    df["dt"] = pd.to_datetime(df["dt"])
    order = np.lexsort((df["dt"].to_numpy(), df["product_id"].to_numpy(), df["store_id"].to_numpy()))
    return Panel(df.iloc[order].reset_index(drop=True), sales[order], oos[order])


def check_oos_orientation(p: Panel) -> dict:
    """Sanity-check that hours_stock_status==1 means OUT of stock (sales should be ~0 there).
    Flips the matrix in place if the data uses the opposite convention."""
    s1 = p.sales[p.oos == 1].mean() if (p.oos == 1).any() else np.nan
    s0 = p.sales[p.oos == 0].mean() if (p.oos == 0).any() else np.nan
    flipped = bool(np.isfinite(s1) and np.isfinite(s0) and s1 > s0)
    if flipped:
        p.oos = (1 - p.oos).astype(np.int8)
    op_cnt = p.oos[:, OP_HOURS].sum(1)
    agree = float((op_cnt == p.df["stock_hour6_22_cnt"].to_numpy()).mean())
    return {"mean_sales_when_flag1": float(s1), "mean_sales_when_flag0": float(s0),
            "flipped": flipped, "share_rows_matching_stock_hour6_22_cnt": agree}


def add_day_index(train: Panel, eval_: Panel) -> int:
    """Integer day index shared by train and eval; returns the last train day (the cutoff)."""
    d0 = train.df["dt"].min()
    for p in (train, eval_):
        p.df["day"] = (p.df["dt"] - d0).dt.days.astype(int)
    return int(train.df["day"].max())


def concat(a: Panel, b: Panel) -> Panel:
    return Panel(pd.concat([a.df, b.df], ignore_index=True),
                 np.vstack([a.sales, b.sales]), np.vstack([a.oos, b.oos]))

In [ ]:
%%writefile frn/metrics.py
"""WAPE / WPE (bias) / VN1 - same definitions as the FreshRetailNet paper and the JD.
error = forecast - actual; WPE < 0 means UNDER-forecasting (the stock-out trap)."""
import numpy as np


def _c(y, f):
    y, f = np.asarray(y, float), np.asarray(f, float)
    m = ~(np.isnan(y) | np.isnan(f))
    return y[m], f[m]


def wape(y, f):
    y, f = _c(y, f)
    return float(np.abs(f - y).sum() / max(y.sum(), 1e-12))


def wpe(y, f):
    y, f = _c(y, f)
    return float((f - y).sum() / max(y.sum(), 1e-12))


def vn1(y, f):
    return wape(y, f) + abs(wpe(y, f))


def summary(y, f):
    return {"wape": wape(y, f), "wpe_bias": wpe(y, f), "vn1": vn1(y, f), "n": int(len(_c(y, f)[0]))}

In [ ]:
%%writefile frn/recovery.py
"""Latent demand recovery: estimate what customers WANTED on days the shelf ran empty.

Observed daily sales on a stock-out day = demand in the in-stock hours only, so
training a forecaster on raw sales teaches it to under-forecast (the paper reports
-7.37% bias). Three estimators, from simplest to strongest:

  observed      sales as recorded (the biased baseline)
  profile       scale the in-stock hours up by the share of a normal day's demand
                they usually carry:  demand ~= observed / sum(profile[in-stock hours])
  model         LightGBM trained SELF-SUPERVISED: take fully in-stock days, hide hours
                using real stock-out patterns copied from real stock-out days, and
                learn to predict the full day from the partial day

How we know recovery works without ever seeing true demand: the same masking trick
on held-out series gives days where the "lost" hours are hidden but known.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from .data import KEY, OP_HOURS, Panel
from .metrics import summary

PROFILE_LEVELS = [KEY, ["store_id", "third_category_id"], ["third_category_id"]]
MIN_SHARE = 0.15       # below this in-stock share, the profile ratio is too unstable


def clean_days(p: Panel) -> np.ndarray:
    return p.oos[:, OP_HOURS].sum(1) == 0


# ------------------------------------------------------------------ hourly profiles
def hourly_profiles(p: Panel, fit_mask: np.ndarray, min_days: int = 5) -> np.ndarray:
    """(n, 24) expected share of the day's demand in each hour, per row, learned from
    clean days only. Falls back store-product -> store-category -> category -> global."""
    n = len(p.df)
    prof = np.full((n, 24), np.nan)
    for level in PROFILE_LEVELS:
        g = p.df.groupby(level, sort=False).ngroup().to_numpy()
        G = g.max() + 1
        hsum = np.zeros((G, 24))
        np.add.at(hsum, g[fit_mask], p.sales[fit_mask])
        cnt = np.bincount(g[fit_mask], minlength=G)
        tot = hsum.sum(1, keepdims=True)
        ok = (cnt >= min_days) & (tot[:, 0] > 0)
        gp = np.divide(hsum, tot, out=np.zeros_like(hsum), where=tot > 0)
        need = np.isnan(prof[:, 0]) & ok[g]
        prof[need] = gp[g[need]]
    glob = p.sales[fit_mask].sum(0)
    glob = glob / glob.sum() if glob.sum() > 0 else np.full(24, 1 / 24)
    prof[np.isnan(prof[:, 0])] = glob
    return prof


def series_level(p: Panel, fit_mask: np.ndarray) -> np.ndarray:
    """Mean daily sales of each series over its clean days (leave-one-out for clean rows)."""
    g = p.df.groupby(KEY, sort=False).ngroup().to_numpy()
    daily = p.sales.sum(1)
    s = np.bincount(g[fit_mask], weights=daily[fit_mask], minlength=g.max() + 1)
    c = np.bincount(g[fit_mask], minlength=g.max() + 1).astype(float)
    num = s[g] - np.where(fit_mask, daily, 0)
    den = c[g] - fit_mask.astype(float)
    lvl = np.divide(num, den, out=np.full(len(g), np.nan), where=den > 0)
    return np.where(np.isnan(lvl), np.nanmean(daily[fit_mask]) if fit_mask.any() else 0.0, lvl)


def profile_recover(sales: np.ndarray, oos: np.ndarray, prof: np.ndarray, level: np.ndarray) -> np.ndarray:
    observed = sales.sum(1)
    share_in = (prof * (1 - oos)).sum(1)
    est = np.where(share_in >= MIN_SHARE, observed / np.maximum(share_in, 1e-9), level)
    est = np.where(oos[:, OP_HOURS].sum(1) == 0, observed, est)      # clean days unchanged
    return np.maximum(observed, est)


# ------------------------------------------------------------------ self-supervised model
def _mask_features(p: Panel, sales: np.ndarray, oos: np.ndarray, prof: np.ndarray, level: np.ndarray) -> pd.DataFrame:
    op = oos[:, OP_HOURS]
    any_oos = op.any(1)
    first = np.where(any_oos, op.argmax(1) + 6, -1)
    last = np.where(any_oos, 22 - op[:, ::-1].argmax(1), -1)
    share_in = (prof * (1 - oos)).sum(1)
    observed = sales.sum(1)
    X = pd.DataFrame({
        "observed": observed,
        "share_in": share_in,
        "profile_est": np.where(share_in >= 0.05, observed / np.maximum(share_in, 1e-9), level),
        "level": level,
        "oos_hours": op.sum(1),
        "first_oos_hour": first,
        "last_oos_hour": last,
        "dow": p.df["dt"].dt.dayofweek.to_numpy(),
        "discount": p.df["discount"].to_numpy(),
        "holiday_flag": p.df["holiday_flag"].to_numpy(),
        "activity_flag": p.df["activity_flag"].to_numpy(),
        "avg_temperature": p.df["avg_temperature"].to_numpy(),
        "precpt": p.df["precpt"].to_numpy(),
        "third_category_id": p.df["third_category_id"].astype("category").to_numpy(),
    })
    return X


def transplant_masks(p: Panel, target_rows: np.ndarray, donor_rows: np.ndarray, rng) -> np.ndarray:
    """Copy real stock-out hour patterns (from donor stock-out days) onto clean target days."""
    picks = rng.choice(donor_rows, size=len(target_rows), replace=True)
    return p.oos[picks].copy()


class RecoveryModel:
    def __init__(self, seed: int = 42, params: dict | None = None):
        self.seed = seed
        self.params = dict(objective="tweedie", tweedie_variance_power=1.3, learning_rate=0.05,
                           num_leaves=63, min_child_samples=100, n_estimators=400,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.9, verbose=-1)
        self.params.update(params or {})

    def _synth(self, p, rows, donors, prof, level, rng):
        m = transplant_masks(p, rows, donors, rng)
        s = p.sales[rows] * (1 - m)              # hide the "stock-out" hours
        sub = p.take(rows)
        X = _mask_features(sub, s, m, prof[rows], level[rows])
        y = p.sales[rows].sum(1)                 # the truth we hid
        keep = m[:, OP_HOURS].any(1)
        return X[keep], y[keep], s[keep], m[keep], rows[keep]

    def fit(self, p: Panel, fit_mask: np.ndarray, prof, level):
        import lightgbm as lgb
        rng = np.random.default_rng(self.seed)
        clean = np.where(clean_days(p) & fit_mask)[0]
        donors = np.where(~clean_days(p) & fit_mask)[0]
        X, y, *_ = self._synth(p, clean, donors, prof, level, rng)
        self.model = lgb.LGBMRegressor(**self.params).fit(X, y, categorical_feature=["third_category_id"])
        return self

    def predict(self, p: Panel, prof, level) -> np.ndarray:
        X = _mask_features(p, p.sales, p.oos, prof, level)
        est = np.clip(self.model.predict(X), 0, None)
        observed = p.sales.sum(1)
        est = np.where(clean_days(p), observed, est)
        return np.maximum(observed, est)


# ------------------------------------------------------------------ orchestration
def recover(p: Panel, cutoff_day: int, seed: int = 42, with_model: bool = True) -> pd.DataFrame:
    """Recovered daily demand for every row with day <= cutoff. Columns:
    observed, rec_profile, rec_model (if with_model)."""
    fit = (p.df["day"].to_numpy() <= cutoff_day)
    prof = hourly_profiles(p, clean_days(p) & fit)
    level = series_level(p, clean_days(p) & fit)
    out = pd.DataFrame({"observed": p.sales.sum(1)})
    out["rec_profile"] = profile_recover(p.sales, p.oos, prof, level)
    if with_model:
        m = RecoveryModel(seed).fit(p, fit, prof, level)
        out["rec_model"] = m.predict(p, prof, level)
    return out


def validate_recovery(p: Panel, cutoff_day: int, holdout_frac: float = 0.2, seed: int = 7) -> pd.DataFrame:
    """Masking test on held-out SERIES: hide hours on their clean days using real stock-out
    patterns, recover, and score against the hidden truth (WAPE, bias=WPE, VN1)."""
    rng = np.random.default_rng(seed)
    g = p.df.groupby(KEY, sort=False).ngroup().to_numpy()
    hold_series = rng.choice(np.unique(g), size=max(1, int(holdout_frac * (g.max() + 1))), replace=False)
    hold = np.isin(g, hold_series)
    fit = (p.df["day"].to_numpy() <= cutoff_day)
    clean = clean_days(p)
    prof = hourly_profiles(p, clean & fit & ~hold)
    level = series_level(p, clean & fit)              # a series' own normal level is legitimately known
    model = RecoveryModel(seed).fit(p, fit & ~hold, prof, level)

    rows = np.where(clean & fit & hold)[0]
    donors = np.where(~clean & fit)[0]
    X, y, s_masked, m, kept = model._synth(p, rows, donors, prof, level, np.random.default_rng(seed + 1))
    estimates = {
        "observed (no recovery)": s_masked.sum(1),
        "profile scaling": profile_recover(s_masked, m, prof[kept], level[kept]),
        "self-supervised LightGBM": np.maximum(s_masked.sum(1), np.clip(model.model.predict(X), 0, None)),
    }
    res = pd.DataFrame([{"method": k, **summary(y, v)} for k, v in estimates.items()])
    res["n_masked_days"] = len(y)
    res["avg_hidden_share"] = 1 - s_masked.sum() / max(y.sum(), 1e-9)
    return res

In [ ]:
%%writefile frn/forecast.py
"""7-day-ahead daily forecasting, trained on (a) raw sales vs (b) recovered demand.

Same direct design as the main project: all target lags are shifted >= 7 days, so one
global LightGBM forecasts every day of the next week from one cutoff. Known-in-advance
inputs (discount plan, holiday/activity flags) are not shifted. Weather uses the actual
values as a stand-in for a weather forecast (stated as an assumption).

Scoring follows the paper: only on days with NO stock-out in operating hours, because
only there is the observed sale the true demand.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from .data import KEY, N_OP_HOURS, OP_HOURS
from .metrics import summary

H = 7
LAGS = [7, 14, 21, 28]
CATS = ["city_id", "store_id", "management_group_id", "first_category_id",
        "second_category_id", "third_category_id"]
KNOWN = ["discount", "holiday_flag", "activity_flag", "precpt", "avg_temperature",
         "avg_humidity", "avg_wind_level"]


def build_frame(df: pd.DataFrame, oos: np.ndarray, target: np.ndarray) -> pd.DataFrame:
    """df/oos/target aligned row-wise, covering train+test days. Adds leakage-safe features."""
    d = df.copy()
    d["y"] = target
    d["oos_hours"] = oos[:, OP_HOURS].sum(1)
    d = d.sort_values(KEY + ["day"]).reset_index(drop=True)
    g = d.groupby(KEY, sort=False)["y"]
    for L in LAGS:
        d[f"lag_{L}"] = g.shift(L)
    base = g.shift(H)
    gb = base.groupby([d[k] for k in KEY])
    for w in (7, 14, 28):
        d[f"rmean_{w}"] = gb.transform(lambda s: s.rolling(w, min_periods=1).mean())
    d["rstd_14"] = gb.transform(lambda s: s.rolling(14, min_periods=2).std())
    d["dow_mean"] = d[[f"lag_{L}" for L in LAGS]].mean(axis=1)
    oos_hist = d.groupby(KEY, sort=False)["oos_hours"].shift(H)
    d["oos_rate_14"] = oos_hist.groupby([d[k] for k in KEY]).transform(
        lambda s: s.rolling(14, min_periods=1).mean()) / N_OP_HOURS
    d["dow"] = d["dt"].dt.dayofweek
    d["dom"] = d["dt"].dt.day
    d["disc_depth"] = 1 - d["discount"]
    for c in CATS + ["product_id"]:
        d[c] = d[c].astype("category")
    return d


FEATURES = (CATS + ["product_id"] + KNOWN + [f"lag_{L}" for L in LAGS] +
            ["rmean_7", "rmean_14", "rmean_28", "rstd_14", "dow_mean", "oos_rate_14", "dow", "dom", "disc_depth"])

PARAMS = dict(objective="tweedie", tweedie_variance_power=1.2, learning_rate=0.05, num_leaves=127,
              min_child_samples=100, n_estimators=600, subsample=0.8, subsample_freq=1,
              colsample_bytree=0.8, verbose=-1)


def fit_predict(frame: pd.DataFrame, cutoff_day: int, params: dict | None = None):
    import lightgbm as lgb
    tr = frame[(frame["day"] <= cutoff_day) & frame["lag_28"].notna()]
    te = frame[(frame["day"] > cutoff_day) & (frame["day"] <= cutoff_day + H)]
    m = lgb.LGBMRegressor(**{**PARAMS, **(params or {})})
    m.fit(tr[FEATURES], tr["y"], categorical_feature=[c for c in CATS + ["product_id"]])
    pred = np.clip(m.predict(te[FEATURES]), 0, None)
    return te.index.to_numpy(), pred, m


EVAL_VIEWS = {
    "A": "A. in-stock days only (paper protocol)",
    "B": "B. all days vs raw sales (censored)",
    "C": "C. all days vs recovered demand, LightGBM (censoring-aware)",
    "C2": "C2. all days vs recovered demand, profile (independent check)",
}


def score(actual: np.ndarray, pred: np.ndarray, clean: np.ndarray, sale_group: np.ndarray | None = None,
          actual_recovered: np.ndarray | None = None, actual_recovered_alt: np.ndarray | None = None) -> list[dict]:
    """Three views of the truth, because none is perfect:
    A  in-stock days: truth is exact, but these days are SELECTED - high-demand days are the
       ones that sell out, so A is biased toward low-demand days and flatters under-forecasting.
    B  all days vs raw sales: truth is censored (too low) on stock-out days.
    C  all days vs recovered demand: unbiased if recovery is unbiased (validated by masking)."""
    rows = [{"eval_view": EVAL_VIEWS["A"], **summary(actual[clean], pred[clean])},
            {"eval_view": EVAL_VIEWS["B"], **summary(actual, pred)}]
    if actual_recovered is not None:
        rows.append({"eval_view": EVAL_VIEWS["C"], **summary(actual_recovered, pred)})
    if actual_recovered_alt is not None:
        rows.append({"eval_view": EVAL_VIEWS["C2"], **summary(actual_recovered_alt, pred)})
    if sale_group is not None:
        for gname in ("high-sale", "low-sale"):
            m = clean & (sale_group == gname)
            rows.append({"eval_view": f"A, {gname} series", **summary(actual[m], pred[m])})
            if actual_recovered is not None:
                m2 = sale_group == gname
                rows.append({"eval_view": f"C, {gname} series", **summary(actual_recovered[m2], pred[m2])})
    return rows

In [ ]:
%%writefile frn/pipeline.py
"""End-to-end experiment shared by the Databricks notebooks, the Kaggle notebook and CI."""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import forecast, recovery
from .data import KEY, N_OP_HOURS, OP_HOURS, Panel, add_day_index, check_oos_orientation, concat

TARGETS = {"observed": "raw sales (baseline)", "rec_profile": "recovered: profile scaling",
           "rec_model": "recovered: self-supervised LightGBM"}


def prepare(train: Panel, eval_: Panel) -> tuple[Panel, int, dict]:
    chk = check_oos_orientation(train)
    if chk["flipped"]:
        eval_.oos = (1 - eval_.oos).astype(np.int8)
    last_train_day = add_day_index(train, eval_)
    full = concat(train, eval_)
    order = np.lexsort((full.df["day"].to_numpy(), full.df["product_id"].to_numpy(), full.df["store_id"].to_numpy()))
    return full.take(order), last_train_day, chk


def sale_groups(full: Panel, cutoff: int) -> np.ndarray:
    daily = pd.Series(full.sales.sum(1))
    hist = daily.where(full.df["day"] <= cutoff)
    mean = hist.groupby([full.df[k] for k in KEY]).transform("mean")
    return np.where(mean >= mean.median(), "high-sale", "low-sale")


def run_fold(full: Panel, cutoff: int, with_model: bool = True, log=print):
    rec = recovery.recover(full, cutoff, with_model=with_model)
    observed = rec["observed"].to_numpy()
    after = full.df["day"].to_numpy() > cutoff
    clean = recovery.clean_days(full)
    groups = sale_groups(full, cutoff)
    # censoring-aware truth for the test days: observed on in-stock days, recovered otherwise
    truth_col = "rec_model" if "rec_model" in rec else "rec_profile"
    recovered_truth = rec[truth_col].to_numpy()
    rows, preds = [], {}
    for col, label in TARGETS.items():
        if col not in rec:
            continue
        target = np.where(after, observed, rec[col].to_numpy())     # test days: never used as labels
        frame = forecast.build_frame(full.df, full.oos, target)
        # frame is re-sorted by key/day; full is already in that order
        idx, pred, _ = forecast.fit_predict(frame, cutoff)
        actual = observed[idx]
        for r in forecast.score(actual, pred, clean[idx], groups[idx], recovered_truth[idx],
                                rec["rec_profile"].to_numpy()[idx]):
            rows.append({"cutoff_day": cutoff, "training_target": label, **r})
        preds[col] = pd.DataFrame({"row": idx, "pred": pred})
        log(f"cutoff {cutoff} | {label}: done")
    return pd.DataFrame(rows), rec, preds


def lost_demand_tables(full: Panel, rec: pd.DataFrame, cutoff: int, col: str = "rec_model"):
    col = col if col in rec else "rec_profile"
    d = full.df[["store_id", "first_category_id", "third_category_id", "day"]].copy()
    d["observed"] = rec["observed"].to_numpy()
    d["recovered"] = rec[col].to_numpy()
    d["oos_hours"] = full.oos[:, OP_HOURS].sum(1)
    d = d[d["day"] <= cutoff]
    d["lost"] = d["recovered"] - d["observed"]
    by_cat = (d.groupby("first_category_id")
                .agg(recovered=("recovered", "sum"), lost=("lost", "sum"), oos_hours=("oos_hours", "mean"))
                .assign(lost_share=lambda x: x.lost / x.recovered, availability=lambda x: 1 - x.oos_hours / N_OP_HOURS)
                .reset_index().sort_values("lost", ascending=False))
    by_store = (d.groupby("store_id")
                  .agg(recovered=("recovered", "sum"), lost=("lost", "sum"), oos_hours=("oos_hours", "mean"))
                  .assign(lost_share=lambda x: x.lost / x.recovered, availability=lambda x: 1 - x.oos_hours / N_OP_HOURS)
                  .reset_index().sort_values("lost", ascending=False))
    hours = full.oos[(full.df["day"] <= cutoff).to_numpy()][:, OP_HOURS].mean(0)
    by_hour = pd.DataFrame({"hour": range(OP_HOURS.start, OP_HOURS.stop), "oos_rate": hours})
    total = {"lost_share_of_demand": float(d["lost"].sum() / d["recovered"].sum()),
             "stockout_day_share": float((d["oos_hours"] > 0).mean()),
             "availability_operating_hours": float(1 - d["oos_hours"].mean() / N_OP_HOURS)}
    return total, by_cat, by_store, by_hour

In [ ]:
open("frn/__init__.py", "w").write("")
import glob, os, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from frn import data, pipeline, recovery
from frn.data import OP_HOURS

# ---- settings
N_SERIES = 20000          # of 50,000 (0 = all; needs ~20+ GB RAM and a long run)
# data: a Kaggle input copy if attached, otherwise download from Hugging Face (Internet ON in settings)
found = glob.glob("/kaggle/input/**/train.parquet", recursive=True)
if os.environ.get("FRN_DATA_DIR"):
    DATA_DIR = os.environ["FRN_DATA_DIR"]
elif found:
    DATA_DIR = os.path.dirname(found[0])
else:
    DATA_DIR = "/kaggle/working/frn_data"
    os.environ.setdefault("HF_HOME", "/kaggle/working/hf")
    data.download(DATA_DIR)
print("data:", DATA_DIR)
keys = data.sample_series(f"{DATA_DIR}/train.parquet", N_SERIES or None, seed=42)
full, last_day, chk = pipeline.prepare(data.load_panel(f"{DATA_DIR}/train.parquet", keys),
                                       data.load_panel(f"{DATA_DIR}/eval.parquet", keys))
print(f"{len(full.df):,} rows | train days 0..{last_day} | orientation check: {chk}")

## 1. When do shelves run empty?
First a data-quality check: hours flagged out-of-stock should show close to zero sales. The code verifies the flag's direction rather than assuming it.

In [ ]:
clean = recovery.clean_days(full)
tr = (full.df["day"] <= last_day).to_numpy()
print(f"stock-out days: {(~clean[tr]).mean():.1%} | availability 06:00-21:59: {1-full.oos[tr][:, OP_HOURS].mean():.1%}")
prof = full.sales[tr & clean].sum(0); prof = prof / prof.sum()
fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.bar(range(24), prof * 100, color="#9db7d5", label="share of daily sales (in-stock days)")
ax2 = ax1.twinx(); ax2.plot(range(24), full.oos[tr].mean(0) * 100, "o-", color="#c0392b", label="out-of-stock rate")
ax1.set_xlabel("hour"); ax1.set_ylabel("% of daily sales"); ax2.set_ylabel("% out of stock")
ax1.set_title("Demand peaks vs when shelves are empty"); fig.legend(loc="upper left", bbox_to_anchor=(.1, .9)); plt.show()

## 2. Recover latent demand, and prove it works
**Masking test:** on held-out series, take in-stock days, hide hours using *real* stock-out patterns copied from real stock-out days, recover, and compare with the demand we hid.
`observed` shows how much demand the stock-out patterns hide. A good method has bias ≈ 0 and the lowest WAPE.

In [ ]:
val = recovery.validate_recovery(full, last_day)
val

## 3. Train the forecaster on raw vs recovered demand
The same global LightGBM (7-day horizon, lags ≥ 7, known discount, holiday and activity plan) is trained with three different targets, in two folds.

**Evaluating it is the subtle part.** In-stock days are not a random sample: high-demand days are the ones that sell out.
So **view A** (in-stock days only, the paper's protocol) skews towards low-demand days and flatters a model that under-forecasts.
**View C** scores all days against recovered demand, with **C2** using the simpler profile recovery as an independent check.
On a simulation where true demand was known, view C tracked the true bias within about 1 point, while view A reversed the ranking.

In [ ]:
results = []
for cutoff in (last_day - 7, last_day):
    res, rec, _ = pipeline.run_fold(full, cutoff)
    results.append(res)
results = pd.concat(results, ignore_index=True)
main = results[results.eval_view.str.match(r"^(A|B|C|C2)\.")]
print("BIAS (WPE): negative = under-forecast")
display(main.pivot_table(index=["cutoff_day", "training_target"], columns="eval_view", values="wpe_bias").round(4))
print("WAPE")
display(main.pivot_table(index=["cutoff_day", "training_target"], columns="eval_view", values="wape").round(4))

In [ ]:
results[~results.eval_view.str.match(r"^(A|B|C|C2)\.")].round(4)

## 4. How much demand do empty shelves cost?

In [ ]:
total, by_cat, by_store, by_hour = pipeline.lost_demand_tables(full, rec, last_day)
print({k: round(v, 4) for k, v in total.items()})
display(by_cat.round(4))
top = by_store.head(max(1, len(by_store) // 5))
print(f"top 20% of stores hold {top.lost.sum() / by_store.lost.sum():.0%} of lost demand")

## Conclusions
*(Fill these in from your own run. The numbers depend on the data.)*

1. **Lost demand:** about __% of demand is invisible in sales because of stock-outs, concentrated in __ and in the __ hours.
2. **Recovery works:** on the masking test, self-supervised LightGBM recovers hidden demand with __% WAPE and __% bias, against __% bias with no recovery.
3. **Forecasts trained on raw sales under-forecast** by __% (view C). Training on recovered demand cuts that to __%.
4. **Evaluation matters:** scoring only in-stock days (view A) gives __, which is a selection effect, not a better model.

**Business takeaway:** a store that forecasts from raw sales keeps re-learning its own shortages. Hourly shelf-availability data is what makes the loop breakable.